In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, 
    mean_absolute_percentage_error, explained_variance_score,
    max_error, median_absolute_error
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow import keras
from tensorflow.keras import layers, optimizers, callbacks
import joblib
import os
import optuna
from optuna.samplers import TPESampler
import time
from datetime import timedelta

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the data
# Identify categorical and numerical columns
categorical_cols = ['current_stop_name', 'next_stop_name', 'day_of_week', 'weather_condition']
numerical_cols = ['is_holiday', 'is_peak_hour', 'passenger_count', 'current_speed', 
                  'distance_to_next_stop', 'current_lat', 'current_lon']

# Create preprocessors
numerical_scaler = StandardScaler()
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform
X_train_num = numerical_scaler.fit_transform(X_train[numerical_cols])
X_test_num = numerical_scaler.transform(X_test[numerical_cols])

X_train_cat = categorical_encoder.fit_transform(X_train[categorical_cols])
X_test_cat = categorical_encoder.transform(X_test[categorical_cols])

# Combine numerical and categorical features
X_train_processed = np.hstack((X_train_num, X_train_cat))
X_test_processed = np.hstack((X_test_num, X_test_cat))

# Reshape data for CNN (add channel dimension)
X_train_cnn = X_train_processed.reshape(X_train_processed.shape[0], X_train_processed.shape[1], 1)
X_test_cnn = X_test_processed.reshape(X_test_processed.shape[0], X_test_processed.shape[1], 1)

# Enhanced evaluation function with comprehensive metrics
def evaluate_model(y_true, y_pred, model_name="Model"):
    # 1. Primary Regression Metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    # Handle potential zero values in y_true for MAPE calculation
    y_true_no_zeros = np.where(y_true == 0, 1e-10, y_true)  # Replace zeros with small value
    mape = calculate_mape(y_true, y_pred, minimum_threshold=0.5)
    
    explained_var = explained_variance_score(y_true, y_pred)
    max_err = max_error(y_true, y_pred)
    median_abs_err = median_absolute_error(y_true, y_pred)
    
    # 3. Error Distribution Metrics
    abs_errors = np.abs(y_true - y_pred)
    p90 = np.percentile(abs_errors, 90)
    p95 = np.percentile(abs_errors, 95)
    p99 = np.percentile(abs_errors, 99)
    
    # 5. Residual Analysis Metrics
    residuals = y_true - y_pred
    mean_residuals = residuals.mean()
    std_residuals = residuals.std()
    
    # Print formatted metrics
    print(f"\n{model_name} Performance Metrics:")
    print(f"{'=' * 50}")
    print("\n1. Primary Regression Metrics:")
    print(f"   MAE: {mae:.4f}")
    print(f"   MSE: {mse:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAPE: {mape:.4f}%")
    print(f"   R2 Score: {r2:.4f}")
    print(f"   Explained Variance: {explained_var:.4f}")
    print(f"   Max Error: {max_err:.4f}")
    print(f"   Median Absolute Error: {median_abs_err:.4f}")
    
    print("\n2. Error Distribution Metrics:")
    print(f"   90th Percentile Error (P90): {p90:.4f}")
    print(f"   95th Percentile Error (P95): {p95:.4f}")
    print(f"   99th Percentile Error (P99): {p99:.4f}")
    
    print("\n3. Residual Analysis Metrics:")
    print(f"   Mean of Residuals: {mean_residuals:.4f}")
    print(f"   Standard Deviation of Residuals: {std_residuals:.4f}")
    
    # Create a dictionary of all metrics
    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2,
        'Explained_Variance': explained_var,
        'Max_Error': max_err,
        'Median_Absolute_Error': median_abs_err,
        'P90': p90,
        'P95': p95,
        'P99': p99,
        'Mean_Residuals': mean_residuals,
        'Std_Residuals': std_residuals
    }
    
    return metrics

def calculate_mape(y_true, y_pred, minimum_threshold=0.5):
    # Filter out values below threshold to avoid division by very small numbers
    mask = y_true >= minimum_threshold
    if np.sum(mask) > 0:
        # Only calculate MAPE for values above threshold
        mape = mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100
    else:
        # If no values above threshold, use alternative metric or default value
        mape = np.nan  # or set a default like 100.0
    return mape

# Function to evaluate train vs test metrics for overfitting analysis
def evaluate_overfitting(model, X_train_data, y_train_data, X_test_data, y_test_data, model_name="Model"):
    # Predict on both train and test
    y_train_pred = model.predict(X_train_data).flatten()
    y_test_pred = model.predict(X_test_data).flatten()
    
    # Calculate metrics for both
    train_mae = mean_absolute_error(y_train_data, y_train_pred)
    test_mae = mean_absolute_error(y_test_data, y_test_pred)
    
    train_mse = mean_squared_error(y_train_data, y_train_pred)
    test_mse = mean_squared_error(y_test_data, y_test_pred)
    
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    
    train_r2 = r2_score(y_train_data, y_train_pred)
    test_r2 = r2_score(y_test_data, y_test_pred)
    
    # Calculate overfitting ratios
    mae_ratio = train_mae / test_mae if test_mae > 0 else float('inf')
    rmse_ratio = train_rmse / test_rmse if test_rmse > 0 else float('inf')
    r2_ratio = train_r2 / test_r2 if test_r2 > 0 else float('inf')
    
    # Adjust ratio interpretation for R2 (closer to 1 is better)
    r2_ratio = 1 / r2_ratio if r2_ratio > 1 else r2_ratio
    
    print(f"\n4. Overfitting Analysis for {model_name}:")
    print(f"   Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}, Ratio: {1/mae_ratio:.4f}")
    print(f"   Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}, Ratio: {1/rmse_ratio:.4f}")
    print(f"   Train R2: {train_r2:.4f}, Test R2: {test_r2:.4f}, Ratio: {r2_ratio:.4f}")
    print(f"   Interpretation: {'Potential overfitting' if 1/rmse_ratio < 0.85 else 'Good generalization'}")
    
    return {
        'Train_MAE': train_mae,
        'Test_MAE': test_mae,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'MAE_Ratio': 1/mae_ratio,
        'RMSE_Ratio': 1/rmse_ratio,
        'R2_Ratio': r2_ratio
    }

# Define baseline CNN model
def create_baseline_cnn_model(input_shape):
    model = keras.Sequential([
        layers.Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.MaxPooling1D(pool_size=2),
        layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Flatten(),
        layers.Dense(50, activation='relu'),
        layers.Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Train baseline CNN model with timing
print("Training baseline CNN model...")
baseline_model = create_baseline_cnn_model((X_train_cnn.shape[1], 1))

# Use early stopping to prevent overfitting
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Start timing
baseline_start_time = time.time()

baseline_history = baseline_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

baseline_training_time = time.time() - baseline_start_time
print(f"\nBaseline model training time: {timedelta(seconds=baseline_training_time)}")

# Evaluate baseline model
print("\nBaseline Model Performance:")
y_pred_baseline = baseline_model.predict(X_test_cnn).flatten()
baseline_metrics = evaluate_model(y_test, y_pred_baseline, "Baseline CNN")

# Evaluate overfitting for baseline model
baseline_overfitting_metrics = evaluate_overfitting(
    baseline_model, X_train_cnn, y_train, X_test_cnn, y_test, "Baseline CNN"
)



# Optuna optimization for CNN with timing
def objective(trial):
    # Define hyperparameters to optimize
    filters1 = trial.suggest_int('filters1', 16, 128)
    filters2 = trial.suggest_int('filters2', 16, 128)
    kernel_size = trial.suggest_int('kernel_size', 2, 5)
    dense_units = trial.suggest_int('dense_units', 16, 128)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    
    # Build model with the suggested hyperparameters
    model = keras.Sequential([
        layers.Conv1D(filters=filters1, kernel_size=kernel_size, activation='relu', 
                     input_shape=(X_train_cnn.shape[1], 1)),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Conv1D(filters=filters2, kernel_size=kernel_size, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(dropout_rate),
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1)
    ])
    
    # Compile model
    optimizer = optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # Define callbacks
    early_stopping = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    # Train model
    history = model.fit(
        X_train_cnn, y_train,
        epochs=100,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Get the validation loss from the epoch with the best performance
    val_loss = min(history.history['val_loss'])
    
    return val_loss

print("\nStarting Optuna optimization...")
optuna_start_time = time.time()
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)
optuna_time = time.time() - optuna_start_time
print(f"\nOptuna optimization time: {timedelta(seconds=optuna_time)}")

# Get the best hyperparameters
best_params = study.best_params
print("\nBest parameters:", best_params)

# Train optimized model with the best hyperparameters
print("\nTraining optimized model with best parameters...")
optimized_model = keras.Sequential([
    layers.Conv1D(filters=best_params['filters1'], kernel_size=best_params['kernel_size'], 
                 activation='relu', input_shape=(X_train_cnn.shape[1], 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Conv1D(filters=best_params['filters2'], kernel_size=best_params['kernel_size'], 
                 activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(best_params['dropout_rate']),
    layers.Flatten(),
    layers.Dense(best_params['dense_units'], activation='relu'),
    layers.Dropout(best_params['dropout_rate']),
    layers.Dense(1)
])

# Compile model
optimizer = optimizers.Adam(learning_rate=best_params['learning_rate'])
optimized_model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Train model with timing
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

optimized_start_time = time.time()
optimized_history = optimized_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=best_params['batch_size'],
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)
optimized_training_time = time.time() - optimized_start_time
print(f"\nOptimized model training time: {timedelta(seconds=optimized_training_time)}")

# Evaluate optimized model
print("\nOptimized Model Performance:")
y_pred_optimized = optimized_model.predict(X_test_cnn).flatten()
optimized_metrics = evaluate_model(y_test, y_pred_optimized, "Optimized CNN")

# Evaluate overfitting for optimized model
optimized_overfitting_metrics = evaluate_overfitting(
    optimized_model, X_train_cnn, y_train, X_test_cnn, y_test, "Optimized CNN"
)

# Try to implement cross-validation metrics if feasible with Keras models
print("\n2. Cross-Validation Metrics:")
print("Note: Full cross-validation for deep learning models is computationally expensive.")
print("Consider implementing with smaller models or on a subset of data if needed.")

# Compare baseline and optimized models
print("\n6. Performance Comparison:")
print(f"{'=' * 50}")
metrics_comparison = pd.DataFrame({
    'Baseline': baseline_metrics,
    'Optimized': optimized_metrics
})

# Calculate improvements
improvements = {}
for metric in baseline_metrics:
    # For metrics where lower is better
    if metric in ['MAE', 'MSE', 'RMSE', 'MAPE', 'Max_Error', 'Median_Absolute_Error', 'P90', 'P95', 'P99', 'Std_Residuals']:
        diff = baseline_metrics[metric] - optimized_metrics[metric]
        pct_improvement = (diff / baseline_metrics[metric]) * 100 if baseline_metrics[metric] != 0 else float('inf')
        improvements[metric] = f"{diff:.4f} ({pct_improvement:.2f}%)"
    # For metrics where higher is better
    elif metric in ['R2', 'Explained_Variance']:
        diff = optimized_metrics[metric] - baseline_metrics[metric]
        if baseline_metrics[metric] > 0:
            pct_improvement = (diff / baseline_metrics[metric]) * 100
        else:
            # Handle case where baseline metric might be negative or zero
            pct_improvement = float('inf') if diff > 0 else float('-inf')
        improvements[metric] = f"{diff:.4f} ({pct_improvement:.2f}%)"
    # For metrics where closer to zero is better (like Mean_Residuals)
    else:
        diff_abs = abs(baseline_metrics[metric]) - abs(optimized_metrics[metric])
        if abs(baseline_metrics[metric]) > 0:
            pct_improvement = (diff_abs / abs(baseline_metrics[metric])) * 100
        else:
            pct_improvement = float('inf') if diff_abs > 0 else float('-inf')
        improvements[metric] = f"{diff_abs:.4f} ({pct_improvement:.2f}%)"

metrics_comparison['Improvement'] = pd.Series(improvements)

# Display key comparisons
print("Key Metrics Comparison:")
for metric in ['RMSE', 'MAE', 'MAPE', 'R2', 'P95']:
    print(f"{metric}: Baseline: {baseline_metrics[metric]:.4f}, Optimized: {optimized_metrics[metric]:.4f}, {improvements[metric]}")

# Save the full comparison to CSV
metrics_comparison.to_csv('model_metrics_comparison.csv')
print("\nFull metrics comparison saved to model_metrics_comparison.csv")

# Training time comparison
print("\n7. Training Time Metrics:")
print(f"{'=' * 50}")
print(f"Baseline Model Training Time: {timedelta(seconds=baseline_training_time)}")
print(f"Optuna Optimization Time: {timedelta(seconds=optuna_time)}")
print(f"Optimized Model Training Time: {timedelta(seconds=optimized_training_time)}")
print(f"Total Time: {timedelta(seconds=baseline_training_time + optuna_time + optimized_training_time)}")

# Save the optimized model
model_filename = 'optimized_cnn_eta_predictor.keras'  # Use .keras extension for TF 2.x
optimized_model.save(model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Save an alternative format if needed
h5_model_filename = 'optimized_cnn_eta_predictor.h5'  # H5 format is also supported
optimized_model.save(h5_model_filename)
print(f"Optimized model also saved as {h5_model_filename}")

# Save the preprocessors for later use
preprocessor_filename = 'eta_preprocessors.pkl'
joblib.dump({
    'numerical_scaler': numerical_scaler,
    'categorical_encoder': categorical_encoder,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols
}, preprocessor_filename)
print(f"Preprocessors saved as {preprocessor_filename}")

# Save metrics to file for future reference
all_metrics = {
    'baseline': baseline_metrics,
    'optimized': optimized_metrics,
    'baseline_overfitting': baseline_overfitting_metrics,
    'optimized_overfitting': optimized_overfitting_metrics,
    'training_times': {
        'baseline_training_time': baseline_training_time,
        'optuna_time': optuna_time,
        'optimized_training_time': optimized_training_time,
        'total_time': baseline_training_time + optuna_time + optimized_training_time
    }
}

metrics_filename = 'full_model_metrics.pkl'
joblib.dump(all_metrics, metrics_filename)
print(f"All metrics saved to {metrics_filename}")

# Feature importance is not directly available for neural networks like in tree-based models
# Alternative approach: Permutation importance
print("\nNote: Feature importance for CNN models requires additional techniques like permutation importance analysis")

# Add code to implement permutation importance
try:
    from sklearn.inspection import permutation_importance
    
    print("\nCalculating permutation feature importance...")
    # First, we need a prediction function that works with the original features
    def predict_func(X_subset):
        # Process the input data exactly as we did during training
        X_subset_num = numerical_scaler.transform(X_subset[numerical_cols])
        X_subset_cat = categorical_encoder.transform(X_subset[categorical_cols])
        X_subset_processed = np.hstack((X_subset_num, X_subset_cat))
        X_subset_cnn = X_subset_processed.reshape(X_subset_processed.shape[0], X_subset_processed.shape[1], 1)
        return optimized_model.predict(X_subset_cnn).flatten()
    
    # Calculate permutation importance
    r = permutation_importance(predict_func, X_test, y_test, 
                              n_repeats=10, random_state=42, n_jobs=-1)
    
    # Create DataFrame of feature importances
    feature_importance = pd.DataFrame({
        'feature': features,
        'importance': r.importances_mean
    }).sort_values('importance', ascending=False)
    
    print("\nPermutation Feature Importance:")
    print(feature_importance)
    
    # Save feature importance to CSV
    feature_importance.to_csv('cnn_feature_importance.csv', index=False)
    print("Feature importance saved to cnn_feature_importance.csv")
    
except Exception as e:
    print(f"\nCould not calculate permutation importance: {e}")
    print("To implement feature importance for the CNN model, consider using permutation_importance from sklearn.inspection")

# Create a comprehensive performance report
print("\n\nGENERATING COMPREHENSIVE PERFORMANCE REPORT")
print(f"{'=' * 75}")

report = """
# ETA Prediction Model Performance Report

## 1. Model Summary
- Baseline: CNN with fixed hyperparameters
- Optimized: CNN with Optuna-tuned hyperparameters
- Dataset: Transit data with {0} features and {1} samples
- Training split: 80%, Test split: 20%

## 2. Key Performance Metrics

| Metric | Baseline | Optimized | Improvement |
|--------|----------|-----------|-------------|
""".format(len(features), len(X))

for metric in ['RMSE', 'MAE', 'MAPE', 'R2', 'Explained_Variance', 'P90', 'P95', 'P99']:
    report += f"| {metric} | {baseline_metrics[metric]:.4f} | {optimized_metrics[metric]:.4f} | {improvements[metric]} |\n"

report += """
## 3. Overfitting Analysis

| Metric | Baseline (Train/Test) | Optimized (Train/Test) |
|--------|----------------------|------------------------|
"""

report += f"| RMSE | {baseline_overfitting_metrics['Train_RMSE']:.4f} / {baseline_overfitting_metrics['Test_RMSE']:.4f} | {optimized_overfitting_metrics['Train_RMSE']:.4f} / {optimized_overfitting_metrics['Test_RMSE']:.4f} |\n"
report += f"| MAE | {baseline_overfitting_metrics['Train_MAE']:.4f} / {baseline_overfitting_metrics['Test_MAE']:.4f} | {optimized_overfitting_metrics['Train_MAE']:.4f} / {optimized_overfitting_metrics['Test_MAE']:.4f} |\n"
report += f"| R2 | {baseline_overfitting_metrics['Train_R2']:.4f} / {baseline_overfitting_metrics['Test_R2']:.4f} | {optimized_overfitting_metrics['Train_R2']:.4f} / {optimized_overfitting_metrics['Test_R2']:.4f} |\n"

report += """
## 4. Training Times

| Process | Duration |
|---------|----------|
"""

report += f"| Baseline Model Training | {timedelta(seconds=baseline_training_time)} |\n"
report += f"| Optuna Hyperparameter Optimization | {timedelta(seconds=optuna_time)} |\n"
report += f"| Optimized Model Training | {timedelta(seconds=optimized_training_time)} |\n"
report += f"| Total Time | {timedelta(seconds=baseline_training_time + optuna_time + optimized_training_time)} |\n"

report += """
## 5. Best Hyperparameters

The following hyperparameters were found to be optimal after Optuna optimization:
"""

for param, value in best_params.items():
    report += f"- {param}: {value}\n"

report += """
## 6. Conclusion

"""

# Add conclusion text based on performance metrics
if optimized_metrics['RMSE'] < 0.8 * baseline_metrics['RMSE']:
    report += "The optimized model shows **substantial improvement** over the baseline model, with significant reductions in error metrics and improved predictive accuracy."
elif optimized_metrics['RMSE'] < 0.95 * baseline_metrics['RMSE']:
    report += "The optimized model shows **moderate improvement** over the baseline model, with notable reductions in error metrics."
else:
    report += "The optimized model shows **slight improvement** over the baseline model. Further optimization strategies may be worth exploring."

# Write the report to a file
with open('eta_prediction_performance_report.md', 'w') as f:
    f.write(report)

print("\nComprehensive performance report saved to eta_prediction_performance_report.md")

c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\cheng\AppData\Local\Temp\ipykernel_8624\345321193.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_8624\345321193.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.ht

Training baseline CNN model...
Epoch 1/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 2.4962 - mae: 0.8544 - val_loss: 0.3973 - val_mae: 0.4011
Epoch 2/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.4126 - mae: 0.4150 - val_loss: 0.3217 - val_mae: 0.3770
Epoch 3/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3880 - mae: 0.4018 - val_loss: 0.3364 - val_mae: 0.4011
Epoch 4/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3726 - mae: 0.3941 - val_loss: 0.3089 - val_mae: 0.3614
Epoch 5/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3586 - mae: 0.3843 - val_loss: 0.3597 - val_mae: 0.3786
Epoch 6/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3513 - mae: 0.3800 - val_loss: 0.3518 - val_mae: 0.3664
Epoch 7/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3455 - mae: 0.3737 - val_loss: 0.3055 - val_mae: 0.3591
Epoch 8/100
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.3591 - mae: 0.3727 - val_loss: 0.2930 - val_mae: 0.3429
E

[I 2025-05-04 02:51:34,606] A new study created in memory with name: no-name-00da42a8-04cd-4867-853a-d7d17d839a23



4. Overfitting Analysis for Baseline CNN:
   Train MAE: 0.3299, Test MAE: 0.3328, Ratio: 1.0088
   Train RMSE: 0.5320, Test RMSE: 0.5374, Ratio: 1.0100
   Train R2: 0.9661, Test R2: 0.9653, Ratio: 0.9991
   Interpretation: Good generalization

Starting Optuna optimization...


c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2025-05-04 02:53:00,268] Trial 0 finished with value: 0.322753369808197 and parameters: {'filters1': 58, 'filters2': 123, 'kernel_size': 4, 'dense_units': 83, 'learning_rate': 0.0002051338263087451, 'batch_size': 64, 'dropout_rate': 0.3832290311184182}. Best is trial 0 with value: 0.322753369808197.
[I 2025-05-04 02:53:42,743] Trial 1 finished with value: 0.4715620279312134 and parameters: {'filters1': 18, 'filters2': 125, 'kernel_size': 5, 'dense_units': 39, 'learning_rate': 0.0002310201887845295, 'batch_size': 64, 'dropout_rate': 0.21649165607921678}. Best is trial 0 with value: 0.322753369808197.
[I 2025-05-0


Optuna optimization time: 1:26:44.246171

Best parameters: {'filters1': 82, 'filters2': 124, 'kernel_size': 5, 'dense_units': 128, 'learning_rate': 0.0002176513380703961, 'batch_size': 64, 'dropout_rate': 0.13588547271932222}

Training optimized model with best parameters...
Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 4.1736 - mae: 1.1796 - val_loss: 0.6446 - val_mae: 0.4935
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.9199 - mae: 0.5885 - val_loss: 0.4932 - val_mae: 0.4486
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.7598 - mae: 0.5318 - val_loss: 0.4011 - val_mae: 0.4003
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.6779 - mae: 0.5050 - val_loss: 0.3828 - val_mae: 0.3952
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.6379 - mae: 0.4903 - val_loss: 0.3609 - val_mae: 0.3875
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.5914 - mae: 0.4766 - val_loss: 0.3366 - val_mae: 0.37

Optimized model also saved as optimized_cnn_eta_predictor.h5
Preprocessors saved as eta_preprocessors.pkl
All metrics saved to full_model_metrics.pkl

Note: Feature importance for CNN models requires additional techniques like permutation importance analysis

Calculating permutation feature importance...

Could not calculate permutation importance: The 'estimator' parameter of permutation_importance must be an object implementing 'fit'. Got <function predict_func at 0x000001A1AE3A7BE0> instead.
To implement feature importance for the CNN model, consider using permutation_importance from sklearn.inspection


GENERATING COMPREHENSIVE PERFORMANCE REPORT

Comprehensive performance report saved to eta_prediction_performance_report.md
